# Session 1 — BERT Text Classification
**Task:** Binary sentiment classification (positive / negative)  
**Model:** `bert-base-uncased` → `BertForSequenceClassification`  
**Dataset:** SST-2 (Stanford Sentiment Treebank, GLUE benchmark)  
**Metric:** Accuracy

---
### What we'll do — step by step
1. Inspect the raw dataset
2. Understand what the tokenizer produces
3. Build a PyTorch Dataset
4. Load the model
5. Train for 3 epochs
6. Evaluate accuracy
7. Run inference on custom sentences

## Step 1 — Imports & Config

In [1]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

# Device: MPS (Apple Silicon) → CUDA → CPU
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 3
LR         = 2e-5
TRAIN_SIZE = 4000
VAL_SIZE   = 500
SAVE_DIR   = "../../models/05_transformers/bert_classification"

print(f"Device: {DEVICE}")

/Users/sameerkhan/anaconda3/envs/sameerkhan/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## Step 2 — Load & Inspect Dataset
SST-2 is a binary sentiment dataset. Each row has a `sentence` and a `label` (0=negative, 1=positive).

In [2]:
raw = load_dataset("glue", "sst2")
print(raw)

print("\n--- 5 sample rows ---")
for i in range(5):
    row = raw["train"][i]
    label = "positive" if row["label"] == 1 else "negative"
    print(f"[{label}] {row['sentence']}")

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

--- 5 sample rows ---
[negative] hide new secretions from the parental units 
[negative] contains no wit , only labored gags 
[positive] that loves its characters and communicates something rather beautiful about human nature 
[negative] remains utterly satisfied to remain the same throughout 
[negative] on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 


## Step 3 — Tokenizer
The tokenizer converts raw text → `input_ids` + `attention_mask`.  
- `input_ids`: each word/subword mapped to a number from BERT's 30k vocabulary  
- `attention_mask`: 1 = real token, 0 = padding  
- `[CLS]` (101) always first, `[SEP]` (102) always last

In [3]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

sample = "I loved this movie!"
enc = tokenizer(sample, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")

print("Tokens:        ", tokenizer.convert_ids_to_tokens(enc["input_ids"][0])[:10], "...")
print("input_ids:     ", enc["input_ids"][0][:10].tolist(), "...")
print("attention_mask:", enc["attention_mask"][0][:10].tolist(), "...")
print(f"\nShape — input_ids: {enc['input_ids'].shape}, attention_mask: {enc['attention_mask'].shape}")

Tokens:         ['[CLS]', 'i', 'loved', 'this', 'movie', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]'] ...
input_ids:      [101, 1045, 3866, 2023, 3185, 999, 102, 0, 0, 0] ...
attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0] ...

Shape — input_ids: torch.Size([1, 128]), attention_mask: torch.Size([1, 128])


## Step 4 — PyTorch Dataset
Wraps the HuggingFace dataset so PyTorch's DataLoader can batch it.

In [4]:
class SST2Dataset(Dataset):
    def __init__(self, hf_split, tokenizer):
        self.data      = hf_split
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["sentence"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),       # (128,)
            "attention_mask": enc["attention_mask"].squeeze(0),  # (128,)
            "labels":         torch.tensor(item["label"], dtype=torch.long),
        }

train_raw = raw["train"].select(range(TRAIN_SIZE))
val_raw   = raw["validation"].select(range(VAL_SIZE))

train_ds = SST2Dataset(train_raw, tokenizer)
val_ds   = SST2Dataset(val_raw,   tokenizer)

# Inspect one item
item = train_ds[0]
print("input_ids shape:     ", item["input_ids"].shape)
print("attention_mask shape:", item["attention_mask"].shape)
print("label:               ", item["labels"].item())

input_ids shape:      torch.Size([128])
attention_mask shape: torch.Size([128])
label:                0


## Step 5 — DataLoaders & Model
`DataLoader` batches the dataset. `BertForSequenceClassification` adds a 2-class linear head on top of BERT's `[CLS]` token output.

In [5]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model on device:  {DEVICE}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total params:     109,483,778
Trainable params: 109,483,778
Model on device:  mps


## Step 6 — Optimizer & Scheduler
- **AdamW**: Adam with weight decay — standard for fine-tuning transformers  
- **Linear warmup scheduler**: slowly increases LR for first 10% of steps, then linearly decays — prevents early training instability

In [6]:
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {int(0.1 * total_steps)}")

Total training steps: 750
Warmup steps:         75


## Step 7 — Training Loop
Each batch: forward → loss → backward → clip gradients → optimizer step → scheduler step.

In [7]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds  = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, scheduler)
    acc  = evaluate(model, val_loader)
    print(f"Epoch {epoch}/{EPOCHS} | loss: {loss:.4f} | val_acc: {acc:.4f}")

Epoch 1/3 | loss: 0.4312 | val_acc: 0.8900


KeyboardInterrupt: 

## Step 8 — Save Model

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

## Step 9 — Inference
Run the trained model on custom sentences.

In [ ]:
def predict(text, model, tokenizer):
    model.eval()
    enc = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs  = torch.softmax(logits, dim=-1)[0]
    label  = logits.argmax(dim=-1).item()
    labels = {0: "negative", 1: "positive"}
    return {"label": labels[label], "confidence": f"{probs[label].item():.2%}"}


tests = [
    "This movie was absolutely fantastic, I loved every second!",
    "What a waste of time. Terrible acting, boring plot.",
    "It was okay, nothing special but not awful either.",
]
for text in tests:
    result = predict(text, model, tokenizer)
    print(f"[{result['label']} {result['confidence']}]  {text}")